# University Knowledge Base Using Forward Chaining

## Aim
To design and implement a First-Order Logic (FOL)-inspired knowledge base for a university domain and use forward chaining to derive new facts and answer queries.

## Problem statement
Represent university information using predicates such as:

- `Teaches(Professor, Course)`
- `Enrolled(Student, Course)`
- `Advisor(Professor, Student)`

Implement forward-chaining inference and answer:

1. Who teaches the course in which Alice is enrolled?
2. Find all students advised by a professor who teaches AI.

## Learning objectives
- Represent facts as predicates and arguments.
- Express inference rules using variables.
- Apply pattern matching and substitution.
- Repeatedly apply rules until no new facts can be derived.
- Answer queries from the resulting knowledge base.

## 1. Background: facts, rules and forward chaining

A **fact** is a statement stored in the knowledge base. For example:

`Teaches(Professor_A, AI)`

A **rule** describes how existing facts can imply a new fact. Variables are written with a `?` prefix. For example:

`Enrolled(?student, ?course) AND Teaches(?professor, ?course) -> Learns_From(?student, ?professor)`

**Forward chaining** is a data-driven inference method. It starts with known facts, finds facts that satisfy a rule's premises, derives the rule's conclusion, and repeats until no new facts are produced.

This notebook implements a small educational rule engine for predicates represented as tuples. It demonstrates variable matching and substitution; it is not a complete FOL theorem prover.

## 2. Import required library

Only Python's standard library is required. No package installation is necessary.

In [ ]:
from pprint import pprint

print("University Knowledge Base")
print("Standard Python libraries imported successfully.")

## 3. Create the initial university knowledge base

The following facts preserve the university example:

- Professor_A teaches AI.
- Professor_B teaches Machine_Learning.
- Professor_C teaches Database.
- Alice, Rahul and Priya are enrolled in the listed courses.
- Professor_A advises Rahul and Priya.
- Professor_B advises Alice.

Each fact is stored as a tuple. The first element is the predicate name; the remaining elements are its arguments.

In [ ]:
facts = {
    ("Teaches", "Professor_A", "AI"),
    ("Teaches", "Professor_B", "Machine_Learning"),
    ("Teaches", "Professor_C", "Database"),
    ("Enrolled", "Alice", "AI"),
    ("Enrolled", "Rahul", "Machine_Learning"),
    ("Enrolled", "Priya", "Database"),
    ("Advisor", "Professor_A", "Rahul"),
    ("Advisor", "Professor_A", "Priya"),
    ("Advisor", "Professor_B", "Alice")
}

print(f"Initial knowledge base contains {len(facts)} facts.\n")
for fact in sorted(facts):
    print(fact)

## 4. Define the inference rules

**Rule 1 – Course instructor inference**

If a student is enrolled in a course and a professor teaches that same course, infer that the student learns from that professor.

`Enrolled(?student, ?course) AND Teaches(?professor, ?course) -> Learns_From(?student, ?professor)`

**Rule 2 – Advisor of an AI instructor**

If a professor teaches AI and advises a student, infer that the student is advised by an AI instructor.

`Teaches(?professor, AI) AND Advisor(?professor, ?student) -> Advised_By_AI_Teacher(?student, ?professor)`

The second rule captures the assignment's query about students advised by a professor who teaches AI.

In [ ]:
rules = [
    {
        "name": "Course instructor inference",
        "premises": [
            ("Enrolled", "?student", "?course"),
            ("Teaches", "?professor", "?course")
        ],
        "conclusion": ("Learns_From", "?student", "?professor")
    },
    {
        "name": "Students advised by an AI instructor",
        "premises": [
            ("Teaches", "?professor", "AI"),
            ("Advisor", "?professor", "?student")
        ],
        "conclusion": ("Advised_By_AI_Teacher", "?student", "?professor")
    }
]

for rule in rules:
    print(rule["name"])
    print("  IF", " AND ".join(map(str, rule["premises"])))
    print("  THEN", rule["conclusion"])

## 5. Implement predicate matching and variable substitution

The matching function compares a rule pattern with a known fact. A term beginning with `?` is treated as a variable. Matching produces a substitution dictionary, such as `{"?student": "Alice", "?course": "AI"}`.

The substitution function replaces variables in a rule conclusion with their matched values.

In [ ]:
def match_pattern(pattern, fact, substitution=None):
    # Match a predicate pattern against a fact and return variable bindings.
    if substitution is None:
        substitution = {}

    if len(pattern) != len(fact) or pattern[0] != fact[0]:
        return None

    updated = substitution.copy()

    for pattern_term, fact_term in zip(pattern[1:], fact[1:]):
        if isinstance(pattern_term, str) and pattern_term.startswith("?"):
            if pattern_term in updated:
                if updated[pattern_term] != fact_term:
                    return None
            else:
                updated[pattern_term] = fact_term
        elif pattern_term != fact_term:
            return None

    return updated


def apply_substitution(pattern, substitution):
    # Replace variables in a predicate pattern with their bound values.
    return tuple(
        substitution.get(term, term)
        if isinstance(term, str) and term.startswith("?")
        else term
        for term in pattern
    )


example_pattern = ("Enrolled", "?student", "?course")
example_fact = ("Enrolled", "Alice", "AI")
example_substitution = match_pattern(example_pattern, example_fact)

print("Pattern:", example_pattern)
print("Fact:", example_fact)
print("Substitution:", example_substitution)

## 6. Find all substitutions that satisfy a rule's premises

For each premise, the engine searches the current knowledge base for matching facts. It carries variable bindings forward so that variables shared between premises must have consistent values.

In [ ]:
def find_substitutions(premises, knowledge_base):
    # Return all consistent substitutions satisfying every premise.
    substitutions = [{}]

    for premise in premises:
        next_substitutions = []

        for current_substitution in substitutions:
            instantiated_premise = apply_substitution(
                premise, current_substitution
            )

            for fact in knowledge_base:
                new_substitution = match_pattern(
                    instantiated_premise,
                    fact,
                    current_substitution
                )

                if new_substitution is not None:
                    next_substitutions.append(new_substitution)

        substitutions = next_substitutions

        if not substitutions:
            break

    unique = {}
    for substitution in substitutions:
        key = tuple(sorted(substitution.items()))
        unique[key] = substitution

    return list(unique.values())


test_substitutions = find_substitutions(
    [
        ("Enrolled", "?student", "?course"),
        ("Teaches", "?professor", "?course")
    ],
    facts
)

print("Bindings satisfying Rule 1:")
for binding in test_substitutions:
    print(binding)

## 7. Implement the forward-chaining algorithm

The algorithm repeatedly:
1. Checks every rule against the current facts.
2. Finds substitutions that satisfy all rule premises.
3. Applies each substitution to the rule conclusion.
4. Adds any genuinely new conclusion to the knowledge base.
5. Stops when a complete pass adds no new facts.

A set is used to prevent duplicate facts.

In [ ]:
def forward_chain(initial_facts, inference_rules, verbose=True):
    # Apply rules repeatedly until no additional facts can be inferred.
    knowledge_base = set(initial_facts)
    iteration = 0

    while True:
        iteration += 1
        newly_inferred = set()

        for rule in inference_rules:
            substitutions = find_substitutions(
                rule["premises"],
                knowledge_base
            )

            for substitution in substitutions:
                conclusion = apply_substitution(
                    rule["conclusion"],
                    substitution
                )

                if conclusion not in knowledge_base:
                    newly_inferred.add(conclusion)

        if not newly_inferred:
            if verbose:
                print(f"No new facts in iteration {iteration}.")
            break

        knowledge_base.update(newly_inferred)

        if verbose:
            print(f"Iteration {iteration}: inferred {len(newly_inferred)} new fact(s).")
            for fact in sorted(newly_inferred):
                print("  ", fact)

    return knowledge_base


inferred_facts = forward_chain(facts, rules)

print("\nInitial fact count:", len(facts))
print("Final knowledge base fact count:", len(inferred_facts))
print("Newly inferred fact count:", len(inferred_facts - facts))

## 8. Inspect all inferred facts

The final knowledge base contains both the original facts and the conclusions derived by forward chaining.

In [ ]:
print("Derived facts:")
for fact in sorted(inferred_facts - facts):
    print(fact)

## 9. Query 1 – Who teaches the course in which Alice is enrolled?

The forward-chaining rule derives `Learns_From(Student, Professor)` whenever the student is enrolled in a course taught by that professor.

We query the derived relation for Alice.

In [ ]:
student_name = "Alice"

matching_results = sorted(
    fact for fact in inferred_facts
    if fact[0] == "Learns_From" and fact[1] == student_name
)

print(f"Query: Who teaches the course in which {student_name} is enrolled?")

if matching_results:
    for fact in matching_results:
        print(f"{fact[2]} teaches the course in which {student_name} is enrolled.")
else:
    print("No matching professor was found.")

## 10. Query 2 – Which students are advised by a professor who teaches AI?

The second rule derives `Advised_By_AI_Teacher(Student, Professor)` from the `Teaches` and `Advisor` facts.

We query all students in this derived relation.

In [ ]:
matching_results = sorted(
    fact for fact in inferred_facts
    if fact[0] == "Advised_By_AI_Teacher"
)

print("Query: Find students advised by a professor who teaches AI.")

if matching_results:
    for fact in matching_results:
        student, professor = fact[1], fact[2]
        print(f"{student} is advised by {professor}, who teaches AI.")
else:
    print("No matching students were found.")

## 11. Query the knowledge base using a reusable function

This helper supports simple predicate queries with constants or variables. For example, `("Teaches", "?professor", "AI")` returns all professors who teach AI.

In [ ]:
def query_knowledge_base(pattern, knowledge_base):
    # Return facts matching a query pattern, with variable bindings.
    answers = []

    for fact in sorted(knowledge_base):
        substitution = match_pattern(pattern, fact)

        if substitution is not None:
            answers.append((fact, substitution))

    return answers


queries = [
    ("Teaches", "?professor", "AI"),
    ("Enrolled", "?student", "Database"),
    ("Advisor", "Professor_A", "?student")
]

for query in queries:
    print("\nQuery:", query)
    answers = query_knowledge_base(query, inferred_facts)

    if answers:
        for fact, binding in answers:
            print("  Fact:", fact, "| Bindings:", binding)
    else:
        print("  No answers found.")

## 12. Verification tests

The following checks verify the two required queries and the inferred relations for the supplied university facts.

In [ ]:
assert ("Learns_From", "Alice", "Professor_A") in inferred_facts
assert ("Advised_By_AI_Teacher", "Rahul", "Professor_A") in inferred_facts
assert ("Advised_By_AI_Teacher", "Priya", "Professor_A") in inferred_facts
assert ("Advised_By_AI_Teacher", "Alice", "Professor_B") not in inferred_facts

print("All verification tests passed.")

## 13. Add a new fact and run inference again

A knowledge base can be extended with new information. Here, a new student is enrolled in AI. Running forward chaining again should derive that the student learns from Professor_A.

This cell uses a separate copy so that the original example remains unchanged.

In [ ]:
extended_facts = set(facts)
extended_facts.add(("Enrolled", "Neha", "AI"))

extended_knowledge_base = forward_chain(
    extended_facts,
    rules,
    verbose=False
)

new_student_result = ("Learns_From", "Neha", "Professor_A")

print("New fact added:", ("Enrolled", "Neha", "AI"))
print("Was the expected conclusion inferred?",
      new_student_result in extended_knowledge_base)

## 14. Observations

1. University information was represented as predicate-based facts.
2. Rules used variables to describe relationships between students, professors and courses.
3. Pattern matching generated substitutions for variables.
4. Forward chaining repeatedly applied rules and added new facts to the knowledge base.
5. Alice was found to be enrolled in AI, which is taught by Professor_A.
6. Rahul and Priya were found to be advised by Professor_A, who teaches AI.
7. Adding a new enrollment fact allowed the system to infer an additional relationship.

## 15. Conclusion

A university-domain knowledge base was designed using predicate-based facts and inference rules. A forward-chaining mechanism was implemented using Python to match rule premises, generate variable substitutions and derive new facts.

The system successfully answered both required queries: identifying the professor teaching Alice's enrolled course and finding students advised by a professor who teaches AI.

The practical demonstrates how rule-based inference can be used for knowledge representation and query answering in a university information system.

## Implementation scope

This is an educational forward-chaining engine for flat predicates and positive rules. It does not implement the full First-Order Logic language, negation, function symbols, existential quantifier elimination, or conflict resolution for contradictory knowledge.